# Compare predicted result to actual 2026 EDC LV line up

1. Extract list of artist in 2026 line up

In [ ]:
# import
from bs4 import BeautifulSoup
import requests
import pandas as pd
import os

# ─── SET THIS EACH YEAR ───────────────────────────────────────────────────────
current_year = 2026   # The EDC lineup year to compare against
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
edc_url = f'https://lasvegas.electricdaisycarnival.com/lineup/{current_year}'

In [ ]:
artists_data = []
year = current_year
url = edc_url

page = requests.get(url)
soup = BeautifulSoup(page.content, 'html')
# Extract artist names from data-artist-name attributes
artist_tags = soup.select('[data-artist-name]')
artist_names = [tag.get('data-artist-name') for tag in artist_tags]
    
# Add year to each artist record
for artist in artist_names:
    # Clean the artist name: lower case and strip whitespace
    clean_artist = str(artist).lower().strip()
    artists_data.append({'year': year, 'artist': clean_artist})
    
print(f"\nTotal artists collected: {len(artists_data)}")

In [ ]:
# Create DataFrame from collected data
df_edc_actual = pd.DataFrame(artists_data)

# Preview the data
print(f"\nSample data:")
df_edc_actual.head(10)

In [ ]:
compare_dir = f'../compare_to_{current_year}'
os.makedirs(compare_dir, exist_ok=True)

filename = f'{compare_dir}/{current_year}_edc_lineup.csv'
df_edc_actual.to_csv(filename, index=False)
print(f"Saved {len(df_edc_actual)} artists to {filename}")

# Extract artists that were predicted to play in 2026 from prediction csv

In [ ]:
compare_dir = f'../compare_to_{current_year}'
input_path = f"../data/result_prediction/edc_{current_year}_prediction.csv"
output_path = f"{compare_dir}/{current_year}_edc_prediction.csv"

# Extract rows where the "result" column is 1 and save to a new CSV file, which means those artists are predicted to be in the lineup. This will allow us to compare the predicted lineup with the actual lineup we just scraped.
df = pd.read_csv(input_path)
filtered = df[df["result"] == 1]
filtered.to_csv(output_path, index=False)

# Print the number of rows written to the new CSV file
print(f"Wrote {len(filtered)} rows to {output_path}")

In [ ]:
compare_dir = f'../compare_to_{current_year}'
pred_path = f"{compare_dir}/{current_year}_edc_prediction.csv"
act_path = f"{compare_dir}/{current_year}_edc_lineup.csv"
out_path = f"{compare_dir}/{current_year}_results.csv"

df_pred = pd.read_csv(pred_path)
df_act = pd.read_csv(act_path)

# Extract sets for quick lookup
actual_set = set(df_act['artist'].dropna())
pred_set = set(df_pred['artist'].dropna())

# Create a new DataFrame with the two columns
results_df = pd.DataFrame({
    'actual_lineup': pd.Series(df_act['artist'].values),
    'predicted_lineup': pd.Series(df_pred['artist'].values)
})

# Add columns to check if the artist in that row's list appeared in the OTHER list
results_df['actual_in_predicted'] = results_df['actual_lineup'].apply(lambda x: x in pred_set if pd.notna(x) else False)

results_df['predicted_in_actual'] = results_df['predicted_lineup'].apply(lambda x: x in actual_set if pd.notna(x) else False)

results_df.to_csv(out_path, index=False)
print(f"Results saved to {out_path} with {len(results_df)} rows.")

results_df.head(10)

In [42]:
# Calculate totals for actual in predicted
actual_in_pred_true = results_df['actual_in_predicted'].sum()
total_actual = results_df['actual_lineup'].notna().sum()
actual_in_pred_false = total_actual - actual_in_pred_true

print("--- RECALL (How many ACTUAL artists were correctly predicted?) ---")
print(f"True (Predicted correctly): {actual_in_pred_true}")
print(f"False (Missed by model): {actual_in_pred_false}")
print(f"Recall: {(actual_in_pred_true / total_actual) * 100:.2f}%")
print(f"Out of all the {total_actual} actual artists that played, the model successfully guessed {actual_in_pred_true} of them\n")


# Calculate totals for predicted in actual
pred_in_act_true = results_df['predicted_in_actual'].sum()
total_pred = results_df['predicted_lineup'].notna().sum()
pred_in_act_false = total_pred - pred_in_act_true

print("--- PRECISION (How many PREDICTED artists actually showed up?) ---")
print(f"True (Correct prediction): {pred_in_act_true}")
print(f"False (Incorrect prediction): {pred_in_act_false}")
print(f"Precision: {(pred_in_act_true / total_pred) * 100:.2f}%")
print(f"Out of all the {total_pred} artists the model predicted, {pred_in_act_true} of them actually played at EDC 2026\n")

# Calculate F1 Score if appropriate
precision = pred_in_act_true / total_pred if total_pred > 0 else 0
recall = actual_in_pred_true / total_actual if total_actual > 0 else 0
if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0
    
print("--- OVERALL F1 SCORE (Harmonic Mean of Precision & Recall) ---")
print(f"F1 Score: {f1 * 100:.2f}%")

# 1. Artists that actually played AND the model predicted them (True Positives)
correctly_predicted = results_df[results_df['actual_in_predicted'] == True]['actual_lineup'].dropna().tolist()

# 2. Artists that actually played BUT the model missed them (False Negatives)
missed_artists = results_df[results_df['actual_in_predicted'] == False]['actual_lineup'].dropna().tolist()

# 3. Artists the model predicted BUT they didn't actually play (False Positives)
incorrect_predictions = results_df[results_df['predicted_in_actual'] == False]['predicted_lineup'].dropna().tolist()

print(f"=== CORRECTLY PREDICTED ARTISTS ({len(correctly_predicted)}) ===")
print(', '.join(sorted(correctly_predicted)))

print(f"\n=== MISSED BY MODEL ({len(missed_artists)}) ===")
print(', '.join(sorted(missed_artists)))

print(f"\n=== INCORRECT PREDICTIONS ({len(incorrect_predictions)}) ===")
print(', '.join(sorted(incorrect_predictions)))

--- RECALL (How many ACTUAL artists were correctly predicted?) ---
True (Predicted correctly): 41
False (Missed by model): 223
Recall: 15.53%
Out of all the 264 actual artists that played, the model successfully guessed 41 of them

--- PRECISION (How many PREDICTED artists actually showed up?) ---
True (Correct prediction): 41
False (Incorrect prediction): 247
Precision: 14.24%
Out of all the 288 artists the model predicted, 41 of them actually played at EDC 2026

--- OVERALL F1 SCORE (Harmonic Mean of Precision & Recall) ---
F1 Score: 14.86%
=== CORRECTLY PREDICTED ARTISTS (41) ===
1991, above & beyond, adventure club, andrew rayel, argy, armin van buuren, boys noize, cutdwn, dead x, delta heavy, doctor p, eli brown, fisher, hybrid minds, i hate models, ilan bluestone, indira paganotto, interplanetary criminal, joseph capriati, kream, lady faith, level up, lil texas, lilly palmer, liquid stranger, lny tnz, meduza, muzz, nico moreno, noizu, paul oakenfold, paul van dyk, restricted, sev